In [0]:
STORAGE_ACCOUNT = "stlakehousedev225"
CATALOG          = "ecommerce_dev"
BRONZE_SCHEMA    = "bronze"
SILVER_SCHEMA    = "silver"
GOLD_SCHEMA      = "gold"
GOLD_BASE        = f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/"

In [0]:
# SET UNITY CATALOG CONTEXT
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {GOLD_SCHEMA}")
print(f"Using catalog: {CATALOG}.{GOLD_SCHEMA}")

Using catalog: ecommerce_dev.gold


In [0]:
# dim_customer
df_dim_customer = spark.table("ecommerce_dev.silver.customers") \
    .select(
        "customer_id",
        "customer_unique_id",
        "customer_city",
        "customer_state"
    )

df_dim_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .option("path", f"{GOLD_BASE}dim_customer") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.dim_customer")

print(f"gold.dim_customer done — {df_dim_customer.count()} rows")

gold.dim_customer done — 99441 rows


In [0]:
# dim_product
df_dim_product = spark.table("ecommerce_dev.silver.products") \
    .select(
        "product_id",
        "product_category_name",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    )

df_dim_product.write \
    .format("delta") \
    .mode("overwrite") \
    .option("path", f"{GOLD_BASE}dim_product") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.dim_product")

print(f"gold.dim_product done — {df_dim_product.count()} rows")

gold.dim_product done — 32951 rows


In [0]:
# dim_seller
df_dim_seller = spark.table("ecommerce_dev.silver.sellers") \
    .select(
        "seller_id",
        "seller_city",
        "seller_state"
    )

df_dim_seller.write \
    .format("delta") \
    .mode("overwrite") \
    .option("path", f"{GOLD_BASE}dim_seller") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.dim_seller")

print(f"gold.dim_seller done — {df_dim_seller.count()} rows")

gold.dim_seller done — 3095 rows


In [0]:
from pyspark.sql.functions import year, month, dayofmonth, quarter, to_date

# dim_date
df_orders = spark.table("ecommerce_dev.silver.orders")

df_dim_date = df_orders \
    .select(to_date("order_purchase_timestamp").alias("order_date")) \
    .distinct() \
    .withColumn("year", year("order_date")) \
    .withColumn("month", month("order_date")) \
    .withColumn("day", dayofmonth("order_date")) \
    .withColumn("quarter", quarter("order_date")) \
    .dropna(subset=["order_date"])

df_dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .option("path", f"{GOLD_BASE}dim_date") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.dim_date")

print(f"gold.dim_date done — {df_dim_date.count()} rows")

gold.dim_date done — 634 rows


In [0]:
from pyspark.sql.functions import to_date

# Read silver tables
df_orders    = spark.table("ecommerce_dev.silver.orders")
df_items     = spark.table("ecommerce_dev.silver.order_items")
df_payments  = spark.table("ecommerce_dev.silver.payments")
df_reviews   = spark.table("ecommerce_dev.silver.reviews")

# Aggregate payments per order
df_payments_agg = df_payments \
    .groupBy("order_id") \
    .agg({"payment_value": "sum"}) \
    .withColumnRenamed("sum(payment_value)", "total_payment_value")

# Aggregate reviews per order (take average score)
df_reviews_agg = df_reviews \
    .groupBy("order_id") \
    .agg({"review_score": "avg"}) \
    .withColumnRenamed("avg(review_score)", "avg_review_score")

# Build fact table
df_fact_orders = df_orders \
    .join(df_items, "order_id", "left") \
    .join(df_payments_agg, "order_id", "left") \
    .join(df_reviews_agg, "order_id", "left") \
    .select(
        "order_id",
        "customer_id",
        "product_id",
        "seller_id",
        "order_status",
        to_date("order_purchase_timestamp").alias("order_date"),
        "price",
        "freight_value",
        "total_payment_value",
        "avg_review_score"
    )

df_fact_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .option("path", f"{GOLD_BASE}fact_orders") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.fact_orders")

print(f"gold.fact_orders done — {df_fact_orders.count()} rows")

gold.fact_orders done — 113425 rows


In [0]:
gold_tables = ["dim_customer", "dim_product", "dim_seller", "dim_date", "fact_orders"]

print(f"{'Table':<20} {'Rows':>10}")
print("-" * 32)

for table in gold_tables:
    count = spark.sql(f"SELECT COUNT(*) FROM {CATALOG}.{GOLD_SCHEMA}.{table}").collect()[0][0]
    print(f"{table:<20} {count:>10,}")

Table                      Rows
--------------------------------
dim_customer             99,441
dim_product              32,951
dim_seller                3,095
dim_date                    634
fact_orders             113,425
